# 📒 Laboratório 1: Introdução à Programação Concorrente em C

## Disciplina: Programação Concorrente (ICP-361)

### Professora: Silvana Rossetto / Gabriel P. Silva
Instituto de Computação/UFRJ

## Apresentação
O objetivo deste laboratório é apresentar os conceitos de criação e manipulação de processos e *threads*

## 1. Criação de Processos

A seguir apresentamos um exemplo com as principais funções utilizadas para a criação de processos, que são `fork()`, `exec() `e `wait()`.

A família *exec* possui várias funções na biblioteca padrão da linguagem C (<unistd.h>) que servem como interfaces  para a verdadeira chamada de sistema, que é o `execve`. A diferença entre elas não está no que elas fazem (todas substituem o processo atual por um novo programa), mas sim na forma como recebem os parâmetros, como localizam o arquivo executável e como lidam com variáveis de ambiente.

Exemplos:

```
execl("/bin/ls", "ls", "-l", "-a", NULL)`;
```

```
char *args[] = {"ls", "-l", NULL};
execv("/bin/ls", args);`
```

```
execlp("ls", "ls", "-l", NULL); /* Note que usamos apenas "ls" e não "/bin/ls"*/
```

```
char *args[] = {"ls", "-l", NULL};
execvp("ls", args);
```

```
char *env[] = {"MEU_VAR=123", NULL};
execle("/bin/ls", "ls", "-l", NULL, env);
```


In [1]:
%%writefile fork.c
#include <stdio.h>
#include <unistd.h>
#include <sys/wait.h>

int main() {
    pid_t pid = fork(); // Cria um novo processo -------------------------------------------------------------------------------------------

    if (pid < 0) {
        // Falha na criação
        fprintf(stderr, "Falha no fork\n");
        return 1;
    } else if (pid == 0) {
        // Bloco executado apenas pelo processo FILHO
        printf("Sou o processo filho (PID: %d). Meu pai eh %d\n", getpid(), getppid());

        // Exemplo de exec: substituir o processo atual por outro programa
        execlp("ls", "ls", "-l", NULL); -------------------------------------------------------------------------------------------
          } else {
        // Bloco executado apenas pelo processo PAI
        printf("Sou o processo pai (PID: %d). Esperando o filho %d...\n", getpid(), pid);
        wait(NULL); // Bloqueia até o processo filho terminar -------------------------------------------------------------------------------------------
        printf("Processo filho terminou.\n");
    }

    return 0;
}

Writing fork.c


In [2]:
!gcc -o fork fork.c -Wall
!./fork

fork.c: In function ‘main’:
fork.c:18:11: error: expected expression before ‘}’ token
   18 |           } else {
      |           ^
/bin/bash: line 1: ./fork: No such file or directory


## 2. pThreads

A seguir apresentamos os conceitos básicos de programação concorrente utilizando a biblioteca `pThreads`. Funções principais utilizadas são:

  1. Cria uma nova thread.

``` c
int pthread_create(pthread_t *thread, const pthread_attr_t *attr,  void *(*start_routine) (void *), void *arg);
```

**Uso e Explicação:**

Esta função é utilizada para criar uma nova *thread* dentro do processo atual. A nova *thread* começa sua execução invocando a função passada como argumento.

  - thread: Um ponteiro para uma variável do tipo `pthread_t`. Se a criação for bem-sucedida, a função armazenará o identificador (ID) da nova *thread* neste endereço.

  - attr: Um ponteiro para uma estrutura `pthread_attr_t` que define os atributos da thread (como tamanho da pilha, política de escalonamento, etc.). Na maioria dos casos simples, passa-se **NULL** para usar os atributos padrão.

  - start_routine: Um ponteiro para a função que a *thread* irá executar. Essa função obrigatoriamente deve retornar um `void*` e receber um único parâmetro `void*`.

  - arg: O argumento que será passado para a função `start_routine`. Como é um `void*`, você pode passar o endereço de qualquer tipo de dado (um inteiro, uma *struct*, etc.) e fazer o cast correto dentro da função. Se a função não precisar de argumentos, passe **NULL**.
  
  - Retorno: Retorna 0 em caso de sucesso. Se houver erro (por exemplo, falta de recursos do sistema), retorna um código de erro numérico.

  2. Encerra a thread atual.

```c
void pthread_exit(void *retval);
```
**Uso e Explicação:**

Esta função encerra a execução da *thread* que a chamou. É uma forma de finalizar a *thread* de maneira limpa, sem encerrar o processo inteiro (como aconteceria se você chamasse exit() da biblioteca padrão do C).

  - retval: Um ponteiro (`void*`) para o valor de retorno da *thread*. Esse valor fica guardado pelo sistema operacional e pode ser recuperado por outra *thread* que chame `pthread_join` esperando por esta. Se você não precisa retornar nada, pode passar **NULL**.
  
  - Retorno: Esta função não retorna para o chamador, pois a *thread* deixa de existir imediatamente após a chamada.

Nota importante: Se a *thread* principal (a que roda a função main) chamar `pthread_exit`, ela é encerrada, mas o processo em si continua vivo até que todas as outras *threads* criadas também terminem.

  3. Aguarda o término de uma thread específica.

```c
int pthread_join(pthread_t thread, void **retval);
```

**Uso e Explicação:**

Esta função bloqueia a execução da *thread* que a chamou até que a *thread* alvo especificada termine sua execução (ou seja, até que a *thread* alvo chame `pthread_exit` ou retorne de sua função principal). É o equivalente ao `wait()` usado para processos.

  - thread: O identificador (ID) da *thread* que você deseja aguardar (o mesmo ID que foi preenchido por pthread_create).

  - retval: Um ponteiro para um ponteiro (`void**`). Se não for **NULL**, a função `pthread_join` copiará o valor de retorno da *thread* alvo (aquele passado no `pthread_exit`) para o local apontado por `retval`. Isso permite recuperar resultados processados pela `thread`. Se você não se importa com o valor de retorno, pode passar NULL.
  
  - Retorno: Retorna `0` em caso de sucesso e um código de erro caso falhe (por exemplo, se tentar fazer join em uma *thread* que não existe ou que não é *joinable*).

Nota importante: Fazer o `join` também instrui o sistema operacional a liberar os recursos (como a pilha) que a *thread* estava usando. Se você não fizer o `join` em uma *thread* (e não a criar como \textit{detached}), ocorrerá um vazamento de recursos conhecido como *zombie thread*.

In [3]:
!gcc -o teste fork.c
!./teste

fork.c: In function ‘main’:
fork.c:18:11: error: expected expression before ‘}’ token
   18 |           } else {
      |           ^
/bin/bash: line 1: ./teste: No such file or directory


## 3. Atividades Iniciais (Revisão de Exemplos)
### Atividade 1: Criação de threads com `hello.c`

#### **Objetivo**: Mostrar a criação básica de *threads*.

In [4]:
%%writefile hello.c
/* Disciplina: Programacao Concorrente */
/* Profa.: Silvana Rossetto */
/* Laboratório: 1 */
/* Codigo: "Hello World" usando threads em C */

#include <stdio.h>
#include <stdlib.h>
#include <pthread.h>

//--funcao executada pelas threads
void *PrintHello (void *arg) {
   //log da thread
   printf("Alô galera do Fundão!\n");

   pthread_exit(NULL);
}

//--funcao principal do programa
int main(int argc, char* argv[]) {
   int nthreads; //qtde de threads que serao criadas (recebida na linha de comando)

   //verifica se o argumento 'qtde de threads' foi passado e armazena seu valor
   if(argc<2) {
      printf("--ERRO: informe a qtde de threads <%s> <nthreads>\n", argv[0]);
      return 1;
   }
   nthreads = atoi(argv[1]);

   //identificadores das threads no sistema
   pthread_t tid_sistema[nthreads];

   //cria as threads e atribui a tarefa de cada thread
   for(int i=0; i<nthreads; i++) {
      printf("--Cria a thread %d\n", i);
      if (pthread_create(&tid_sistema[i], NULL, PrintHello, NULL)) {
         printf("--ERRO: pthread_create()\n");
         return 2;
      }
   }

   //log da funcao main
   printf("--Thread principal terminou\n");

   pthread_exit(NULL); //necessario para que o processo não seja encerrado antes das threads terminarem
}

Writing hello.c


In [16]:
# Célula de Código: Exemplo de compilação e execução
!gcc -o hello hello.c -Wall -lpthread
!./hello 4

--Cria a thread 0
--Cria a thread 1
--Cria a thread 2
--Cria a thread 3
--Thread principal terminou
Alô galera do Fundão!
Alô galera do Fundão!
Alô galera do Fundão!
Alô galera do Fundão!


Questão: O resultado impresso muda de uma execução para outra?

Sim, devido as thredas executarem ao mesmo tempo, não há como saber quem começa e quem termina primeiro.

### Atividade 2: Passagem de argumentos (helloArg.c)

#### Objetivo: Mostrar como passar *um* argumento simples para uma thread

In [21]:
%%writefile helloArg.c
/* Disciplina: Programacao Concorrente */
/* Profa.: Silvana Rossetto */
/* Laboratório: 1 */
/* Codigo: "Hello World" usando threads em C com passagem de um argumento */

#include <stdio.h>
#include <stdlib.h>
#include <pthread.h>

#define TAMANHOS

//--funcao executada pelas threads
void *PrintHello (void* arg) {
   //typecasting do argumento
   long int idThread = (long int) arg;
   //log da thread
   printf("Hello World da thread: %ld\n", idThread);

   pthread_exit(NULL);
}

//--funcao principal do programa
int main(int argc, char * argv[]) {
   int nthreads; //qtde de threads que serao criadas (recebida na linha de comando)

   //verifica se o argumento 'qtde de threads' foi passado e armazena seu valor
   if(argc<2) {
       printf("--ERRO: informe a qtde de threads <%s> <nthreads>\n", argv[0]);
       return 1;
   }
   nthreads = atoi(argv[1]);

   //identificadores das threads no sistema
   pthread_t tid_sistema[nthreads];

#ifdef TAMANHOS
   //exibe os tamanhos dos tipos - para informacao apenas
   puts("Tamanhos dos tipos de dados:");
   printf("Tamanho ponteiro:%ld bytes\n", sizeof(void*));
   printf("Tamanho int:%ld bytes\n", sizeof(int));
   printf("Tamanho long int:%ld bytes\n\n", sizeof(long int));
#endif

   //cria as threads e atribui a tarefa de cada thread
   for(long int i=1; i<=nthreads; i++) {
      printf("--Cria a thread %ld\n", i);
      if (pthread_create(&tid_sistema[i-1], NULL, PrintHello, (void*) i)) {
         printf("--ERRO: pthread_create()\n");
	 return 2;
      }
   }

   //log da função main
   printf("--Thread principal terminou\n");
   pthread_exit(NULL); //necessario para que o processo não seja encerrado antes das threads terminarem
   //return 0;
}

Overwriting helloArg.c


In [22]:
!gcc -o teste helloArg.c -Wall -lpthread
!./teste 4

Tamanhos dos tipos de dados:
Tamanho ponteiro:8 bytes
Tamanho int:4 bytes
Tamanho long int:8 bytes

--Cria a thread 1
--Cria a thread 2
Hello World da thread: 1
--Cria a thread 3
Hello World da thread: 2
--Cria a thread 4
Hello World da thread: 3
--Thread principal terminou
Hello World da thread: 4


### Questão: Por que foi necessário usar `long int` na variável iteradora?

Por que estamos passando i como argumento em pthread_create, que por sua vez recebe ponteiros de tipo void, então ao usar long int, temos apenas um cast simples de mesmo tamanho evitando truncamento.

### Atividade 3: Múltiplos argumentos (helloArgs.c)

#### Objetivo:
Mostrar o uso de estruturas para múltiplos argumentos.

In [23]:
%%writefile helloArgs.c
/* Disciplina: Programacao Concorrente */
/* Profa.: Silvana Rossetto */
/* Laboratório: 1 */
/* Codigo: "Hello World" usando threads em C passando mais de um argumento */

#include <stdio.h>
#include <stdlib.h>
#include <pthread.h>

//cria a estrutura de dados para armazenar os argumentos da thread
typedef struct {
   int idThread, nThreads;
} t_Args;

//--funcao executada pelas threads
void *PrintHello (void *arg) {
  //typecarting do argumento
  t_Args args = *(t_Args*) arg;

  //log da thread
  printf("Sou a thread %d de %d threads\n", args.idThread, args.nThreads);

  free(arg); //libera a memoria que foi alocada na main

  pthread_exit(NULL);
}

//--funcao principal do programa
int main(int argc, char* argv[]) {
  t_Args *args; //receberá os argumentos para a thread
  int nthreads; //qtde de threads que serao criadas (recebida na linha de comando)

  //verifica se o argumento 'qtde de threads' foi passado e armazena seu valor
  if(argc<2) {
      printf("--ERRO: informe a qtde de threads <%s> <nthreads>\n", argv[0]);
      return 1;
  }
  nthreads = atoi(argv[1]);

  //identificadores das threads no sistema
  pthread_t tid_sistema[nthreads];

  //cria as threads
  for(int i=1; i<=nthreads; i++) {
    printf("--Aloca e preenche argumentos para thread %d\n", i);
    args = malloc(sizeof(t_Args));
    if (args == NULL) {
      printf("--ERRO: malloc()\n");
      return 2;
    }
    args->idThread = i;
    args->nThreads = nthreads;

    printf("--Cria a thread %d\n", i);
    if (pthread_create(&tid_sistema[i-1], NULL, PrintHello, (void*) args)) {
      printf("--ERRO: pthread_create() da thread %d\n", i);
    }
  }

  //log da função principal
  printf("--Thread principal terminou\n");

  pthread_exit(NULL);
}

Overwriting helloArgs.c


In [30]:
!gcc -o teste helloArgs.c -Wall -lpthread
!./teste 4

--Aloca e preenche argumentos para thread 1
--Cria a thread 1
--Aloca e preenche argumentos para thread 2
--Cria a thread 2
Sou a thread 1 de 4 threads
--Aloca e preenche argumentos para thread 3
--Cria a thread 3
Sou a thread 2 de 4 threads
--Aloca e preenche argumentos para thread 4
--Cria a thread 4
--Thread principal terminou
Sou a thread 3 de 4 threads
Sou a thread 4 de 4 threads


### Questão: Por que foi necessário criar uma estrutura de dados nova?

Para poder alocar uma "pacote" em um espaço na heap e na hora de passar a referência em pthread_create, passamos apenas o endereço para depois fazer o "desempacotamento"

### Atividade 4: Sincronização com pthread_join (helloJoin.c)

#### Objetivo: Fazer a thread principal (main) aguardar as threads criadas

In [32]:
%%writefile helloJoin.c
/* Disciplina: Programacao Concorrente */
/* Profa.: Silvana Rossetto */
/* Laboratório: 1 */
/* Codigo: "Hello World" usando threads em C e a funcao que espera as threads terminarem */

#include <stdio.h>
#include <stdlib.h>
#include <pthread.h>

//cria a estrutura de dados para armazenar os argumentos da thread
typedef struct {
   int idThread, nThreads;
} t_Args;

//funcao executada pelas threads
void *PrintHello (void *arg) {
  t_Args *args = (t_Args *) arg;

  printf("Sou a thread %d de %d threads\n", args->idThread, args->nThreads);
  free(arg); //libera a alocacao feita na main

  pthread_exit(NULL);
}

//funcao principal do programa
int main(int argc, char* argv[]) {
  t_Args *args; //receberá os argumentos para a thread

  int nthreads; //qtde de threads que serao criadas (recebida na linha de comando)

  //verifica se o argumento 'qtde de threads' foi passado e armazena seu valor
  if(argc<2) {
      printf("--ERRO: informe a qtde de threads <%s> <nthreads>\n", argv[0]);
      return 1;
  }
  nthreads = atoi(argv[1]);

  //identificadores das threads no sistema
  pthread_t tid_sistema[nthreads];

  //cria as threads
  for(int i=1; i<=nthreads; i++) {
    printf("--Aloca e preenche argumentos para thread %d\n", i);
    args = malloc(sizeof(t_Args));
    if (args == NULL) {
      printf("--ERRO: malloc()\n");
      return 1;
    }
    args->idThread = i;
    args->nThreads = nthreads;

    printf("--Cria a thread %d\n", i);
    if (pthread_create(&tid_sistema[i-1], NULL, PrintHello, (void*) args)) {
      printf("--ERRO: pthread_create()\n");
      return 2;
    }
  }

  //espera todas as threads terminarem
  for (int i=0; i<nthreads; i++) {
    if (pthread_join(tid_sistema[i], NULL)) {
         printf("--ERRO: pthread_join() da thread %d\n", i);
    }
  }

  //log da função principal
  printf("--Thread principal terminou\n");

  //pthread_exit(NULL); //nao é necessario nesse caso, por que?
}

Writing helloJoin.c


In [36]:
!gcc -o teste helloJoin.c -Wall -lpthread
!./teste 4

--Aloca e preenche argumentos para thread 1
--Cria a thread 1
--Aloca e preenche argumentos para thread 2
--Cria a thread 2
--Aloca e preenche argumentos para thread 3
--Cria a thread 3
--Aloca e preenche argumentos para thread 4
--Cria a thread 4
Sou a thread 3 de 4 threads
Sou a thread 1 de 4 threads
Sou a thread 2 de 4 threads
Sou a thread 4 de 4 threads
--Thread principal terminou


### Atividade 5: Implementação Prática

#### Desafio: Implementar um programa concorrente com $M$ threads para somar 1 a cada elemento de um vetor de $N$ elementos.

#### Requisitos: Divisão balanceada de carga entre as threads. $M$ e $N$ passados como argumentos de linha de comando. Implementar funções de inicialização e verificação de resultados.

In [46]:
%%writefile exercicio5.c
#include <stdio.h>
#include <stdlib.h>
#include <pthread.h>

// Defina sua estrutura de dados e função da thread aqui...
typedef struct{
  int idThread;
  int tam_vet;
  int inicio;
  int fim;

} t_Args;

int *vetor;

void* tarefa(void* arg){
  t_Args* args = (t_Args*) arg;
  for(int i = args->inicio; i < args->fim; i++){
    vetor[i]++;
  }
  free(arg);
  pthread_exit(NULL);
}

int main(int argc, char* argv[]){
  int nThreads, tamVet;

  if(argc<3){
    printf("--ERRO: informe a qtde de threads <%s> <nthreads>\n", argv[0]);
    return 1;
  }
  nThreads = atoi(argv[1]);
  tamVet = atoi(argv[2]);

  vetor = malloc(tamVet * sizeof(int));

  for(int i = 0; i < tamVet; i++){
    vetor[i] = 0;
  }

  pthread_t tid[nThreads];

  for(int i = 0; i < nThreads; i++){
    t_Args *args = malloc(sizeof(t_Args));
    args->idThread = i;
    args->tam_vet = tamVet;
    args->inicio = i * (tamVet / nThreads);
    args->fim = (i + 1) * (tamVet / nThreads);
    if(pthread_create(&tid[i], NULL, tarefa, (void*) args)){
      printf("--ERRO: pthread_create()\n");
      return 2;
    }
  }

  for(int i = 0; i < nThreads; i++){
    if(pthread_join(tid[i], NULL)){
      printf("--ERRO: pthread_join()\n");
      return 3;
    }
  }

  int soma = 0;
  for(int i = 0; i < tamVet; i++){
    soma += vetor[i];
  }

  if(soma == tamVet){
    printf("%d == %d\nO vetor está correto\n", soma, tamVet);
  }
  else{
    printf("O vetor não está correto\n");
  }

  free(vetor);
}

/* Alocação de memória */

/* 1. Inicializa o vetor  */

/* 2. Cria as M threads */

/* 3. Aguarda o término das threads */

/* 4. Verifica se o resultado está correto */

/* Libera memória e encerra */


Overwriting exercicio5.c


In [48]:
# Compile o seu programa
!gcc -o exercicio5 exercicio5.c -Wall -lpthread

# Execute com M=4 threads e N=1000 elementos
!./exercicio5 4 1000

1000 == 1000
O vetor está correto


### 3. Entrega
Disponibilize o código no GitHub ou GitLab.